In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import LogisticRegression, LinearSVC
from pyspark.ml.regression import LinearRegression
from pyspark.ml.clustering import KMeans


In [2]:
spark = SparkSession.builder.appName("ML_Model_Testing").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/24 12:51:17 WARN Utils: Your hostname, LAPTOP-H1TA7CV3, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/24 12:51:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 12:51:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/24 12:51:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/24 12:51:18 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
supervised_data = [
    # id, final_features, target_class (binary), target_value (continuous)
    (1, Vectors.dense([ 2.0,  1.1,  0.1]), 1.0,  15.5),
    (2, Vectors.dense([ 2.2,  0.9,  0.2]), 1.0,  16.2),
    (3, Vectors.dense([ 1.8,  1.2,  0.0]), 1.0,  14.8),
    (4, Vectors.dense([ 1.9,  1.0,  0.1]), 1.0,  15.0),
    (5, Vectors.dense([-1.0, -2.1, -0.5]), 0.0, -10.1),
    (6, Vectors.dense([-1.2, -1.9, -0.6]), 0.0, -9.5),
    (7, Vectors.dense([-0.9, -2.2, -0.4]), 0.0, -11.0),
    (8, Vectors.dense([-1.1, -2.0, -0.5]), 0.0, -10.0)
]
# Create the DataFrame
supervised_df = spark.createDataFrame(
    supervised_data, 
    ["id", "final_features", "target_class", "target_value"]
)

# For Regression/Classification, we usually split data into train and test sets
# randomSplit takes a list of weights [train_percentage, test_percentage]
train_df, test_df = supervised_df.randomSplit([0.7, 0.3], seed=42)

print("Supervised Training Data:")
train_df.show(truncate=False)

Supervised Training Data:


+---+----------------+------------+------------+
|id |final_features  |target_class|target_value|
+---+----------------+------------+------------+
|2  |[2.2,0.9,0.2]   |1.0         |16.2        |
|3  |[1.8,1.2,0.0]   |1.0         |14.8        |
|4  |[1.9,1.0,0.1]   |1.0         |15.0        |
|5  |[-1.0,-2.1,-0.5]|0.0         |-10.1       |
|7  |[-0.9,-2.2,-0.4]|0.0         |-11.0       |
+---+----------------+------------+------------+



In [4]:
clustering_data = [
    # Cluster 1 (Tightly packed around coordinates 0,0)
    (1, Vectors.dense([0.1, 0.1])),
    (2, Vectors.dense([0.2, 0.0])),
    (3, Vectors.dense([0.0, 0.2])),
    
    # Cluster 2 (Tightly packed around coordinates 5,5)
    (4, Vectors.dense([5.1, 5.1])),
    (5, Vectors.dense([5.0, 5.2])),
    (6, Vectors.dense([5.2, 5.0])),
    
    # Cluster 3 (Tightly packed around coordinates -5,-5)
    (7, Vectors.dense([-5.1, -5.1])),
    (8, Vectors.dense([-5.0, -5.2])),
    (9, Vectors.dense([-5.2, -5.0]))
]

# Create the DataFrame (Note the DataFrame name 'df' to match the KMeans snippet)
df = spark.createDataFrame(
    clustering_data, 
    ["id", "scaled_features"]
)

print("Unsupervised Clustering Data:")
df.show(truncate=False)

Unsupervised Clustering Data:
+---+---------------+
|id |scaled_features|
+---+---------------+
|1  |[0.1,0.1]      |
|2  |[0.2,0.0]      |
|3  |[0.0,0.2]      |
|4  |[5.1,5.1]      |
|5  |[5.0,5.2]      |
|6  |[5.2,5.0]      |
|7  |[-5.1,-5.1]    |
|8  |[-5.0,-5.2]    |
|9  |[-5.2,-5.0]    |
+---+---------------+



In [5]:
lr = LogisticRegression(
    featuresCol="final_features",
    labelCol="target_class",
    maxIter=10,
    regParam=0.3,
    elasticNetParam=0.8
)
lr_model = lr.fit(train_df)

lin_reg = LinearRegression(
    featuresCol="final_features",
    labelCol="target_value",
    maxIter=10,
    regParam=0.3,
)
lin_model = lin_reg.fit(train_df)

predictions = lr_model.transform(test_df)

26/06/24 12:54:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/24 12:54:31 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [ ]:
from pyspark.ml.classification import LinearSVC

svm = LinearSVC(
    featuresCol="final_features", 
    labelCol="target_class", 
    maxIter=50, 
    regParam=0.1
)

# Fit the model
svm_model = svm.fit(train_df)

# View the coefficients (the mathematical definition of the hyperplane)
print(f"Coefficients: {svm_model.coefficients}")
print(f"Intercept: {svm_model.intercept}")
spark.stop()

Coefficients: [0.2927773965919302,0.27327437038359953,1.4424873361678439]
Intercept: 0.14508103101120418


26/06/24 13:27:28 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1576094 ms exceeds timeout 120000 ms
26/06/24 13:27:28 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/24 13:27:32 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1530)
	at 